1. Instalasi & Import Library

In [11]:
import pandas as pd
import numpy as np
import glob
import os


2. Load & Merge Dataset Mentah

In [ ]:
# Menggunakan '../' karena notebook ini berada di folder 'notebooks'
path_raw = '../data/raw_x/'
all_files = glob.glob(os.path.join(path_raw, "*.csv"))

df_list = []
for file in all_files:
    try:
        df_temp = pd.read_csv(file)
        
        # Ekstrak nama file (tanpa .csv) untuk tahu dari file mana data ini berasal
        kategori_asal = os.path.basename(file).replace('.csv', '')
        df_temp['kategori_sumber'] = kategori_asal
        
        df_list.append(df_temp)
    except Exception as e:
        print(f"Gagal membaca file {file}: {e}")

# Gabungkan semua DataFrame menjadi satu
df_raw = pd.concat(df_list, ignore_index=True)

print(f"Total dataset mentah yang berhasil di-load: {df_raw.shape[0]} baris.")
display(df_raw.head(3))

Total dataset mentah yang berhasil di-load: 1873 baris.


,web_scraper_order,web_scraper_start_url,data,data2,data3,data4,data5,data6,data8,kategori_sumber,data7,data9,data10,data12,data11,data13
0,1775290311-1,https://x.com/search?q=%22PDAM%20mati%22%20OR%...,8m,Koh Alung,@alung_koh21471,krisis air,Pemerintah berupaya menangani,bersih di Karang Baru secara bertahap.,3,Air & Sanitasi,NaN,NaN,NaN,NaN,NaN,NaN
1,1775290311-2,https://x.com/search?q=%22PDAM%20mati%22%20OR%...,16m,Batampos,@BatamPos,"hingga Hari ke -70, Warga Tanjung Sengkuang Ma...",Krisis Air,NaN,5,Air & Sanitasi,NaN,NaN,NaN,NaN,NaN,NaN
2,1775290311-3,https://x.com/search?q=%22PDAM%20mati%22%20OR%...,17m,Rizki Fadillnaldo,@ciroalves2055,KRISIS AIR,NaN,BERSIH? JANGAN ASAL MENYALAHKAN!\nMasalahnya l...,3,Air & Sanitasi,NaN,NaN,NaN,NaN,NaN,NaN


3. Menyatukan Teks yang Terpecah

In [ ]:
print("Memulai proses pembersihan dan penyatuan teks...")

# Identifikasi kolom waktu (biasanya bernama 'data')
time_col = 'data' if 'data' in df_raw.columns else None

# Ambil semua kolom teks hasil scraper (berawalan 'data' tapi bukan kolom waktu)
text_cols = [c for c in df_raw.columns if c.startswith('data') and c != time_col]

def bersihkan_dan_gabung(row):
    teks_gabungan = []
    username = 'Tidak diketahui'
    
    for col in text_cols:
        val = str(row[col]) if pd.notna(row[col]) else ''
        val = val.strip()
        if not val:
            continue
            
        # Jika kata berawalan @ dan berdiri sendiri, jadikan username
        if val.startswith('@') and len(val.split()) == 1:
            username = val
        # Jika bukan spam angka (seperti jumlah like/retweet), masukkan ke teks
        elif not (val.isdigit() and len(val) < 4):
            teks_gabungan.append(val)
            
    return pd.Series([username, ' '.join(teks_gabungan)])

# Terapkan fungsi ke seluruh baris
df_raw[['username', 'teks_bersih']] = df_raw.apply(bersihkan_dan_gabung, axis=1)

# Buat dataframe baru untuk data interim
df_clean = df_raw.copy()

# Pindahkan kolom waktu
if time_col:
    df_clean['waktu'] = df_clean[time_col]
else:
    df_clean['waktu'] = 'Tidak diketahui'

# Buang baris yang teksnya kosong setelah dibersihkan
df_clean = df_clean[df_clean['teks_bersih'] != '']

print(f"Total baris setelah teks disatukan & dibersihkan: {df_clean.shape[0]}")
display(df_clean[['waktu', 'username', 'teks_bersih']].head(3))

Memulai proses pembersihan dan penyatuan teks...
Total baris setelah teks disatukan & dibersihkan: 1863


,waktu,username,teks_bersih
0,8m,@alung_koh21471,Koh Alung krisis air Pemerintah berupaya menan...
1,16m,@BatamPos,"Batampos hingga Hari ke -70, Warga Tanjung Sen..."
2,17m,@ciroalves2055,Rizki Fadillnaldo KRISIS AIR BERSIH? JANGAN AS...


4. Restrukturisasi Kolom Target ADUIN

In [ ]:
print("Menyusun arsitektur kolom...")

# 1. Jadikan asal file sebagai kolom kategori utama
df_clean['kategori_isu'] = df_clean['kategori_sumber']

# 2. Setup Kolom Target NLP dengan nilai Placeholder
df_clean['urgensi'] = 'Belum Dilabeli'
df_clean['sentimen'] = 'Belum Dilabeli'
df_clean['lokasi_insiden'] = 'Tidak diketahui'

# 3. Restrukturisasi Urutan Kolom Akhir
kolom_urut = ['waktu', 'username', 'teks_bersih', 'kategori_isu', 'urgensi', 'sentimen', 'lokasi_insiden']
df_aduin = df_clean[kolom_urut].copy()

# Hapus duplikat berdasarkan teks keluhan agar data benar-benar unik
df_aduin = df_aduin.drop_duplicates(subset=['username', 'waktu', 'teks_bersih']).reset_index(drop=True)

print("Blueprint tabel ADUIN berhasil diperbarui!")
display(df_aduin.head(3))

# Cek kembali
print("\n--- Pengecekan Distribusi Kategori ---")
print(df_aduin['kategori_isu'].value_counts())

Menyusun arsitektur kolom...
Blueprint tabel ADUIN berhasil diperbarui!


,waktu,username,teks_bersih,kategori_isu,urgensi,sentimen,lokasi_insiden
0,8m,@alung_koh21471,Koh Alung krisis air Pemerintah berupaya menan...,Air & Sanitasi,Belum Dilabeli,Belum Dilabeli,Tidak diketahui
1,16m,@BatamPos,"Batampos hingga Hari ke -70, Warga Tanjung Sen...",Air & Sanitasi,Belum Dilabeli,Belum Dilabeli,Tidak diketahui
2,17m,@ciroalves2055,Rizki Fadillnaldo KRISIS AIR BERSIH? JANGAN AS...,Air & Sanitasi,Belum Dilabeli,Belum Dilabeli,Tidak diketahui



--- Pengecekan Distribusi Kategori ---
kategori_isu
Infrastruktur & Jalan    416
Ekonomi & UMKM           388
Lingkungan & Sampah      176
Pelayanan Publik         163
Pendidikan               156
Air & Sanitasi           145
Keamanan & Sosial        128
Transportasi             106
Bencana & Cuaca          103
Kesehatan                 37
Name: count, dtype: int64


5. Ekspor ke Folder Processed

In [16]:
import os

output_folder = '../data/processed/'
output_filename = 'twitter_keluhan_processed_v2.csv'
final_path = os.path.join(output_folder, output_filename)

if not os.path.exists(output_folder):
    os.makedirs(output_folder)
    
df_aduin.to_csv(final_path, index=False)

print(f"Sukses! Data dengan format 1 kolom kategori berhasil diekspor ke: {final_path}")

Sukses! Data dengan format 1 kolom kategori berhasil diekspor ke: ../data/processed/twitter_keluhan_processed_v2.csv
